In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q langchain-text-splitters

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import json
from tqdm import tqdm
import numpy as np
import pandas as pd
import time
import torch

# Read the documents in MIRACL Persian

In [ ]:
import json


data = []
with open('/content/drive/MyDrive/docs-0.jsonl', 'r', encoding='utf8') as f:
    for line in f:
        data.append(json.loads(line.strip()))

print(f"Loaded {len(data)} records")

Loaded 500000 records


In [ ]:
data[1]

{'docid': '594#1',
 'title': 'ویکی\u200cپدیا',
 'text': 'ویکی\u200cپدیای انگلیسی در تاریخ ۱۵ ژانویه ۲۰۰۱ (۲۶ دی ۱۳۷۹) به صورت مکملی برای دانشنامهٔ تخصصی نیوپدیا نوشته شد. بنیان\u200cگذاران آن «جیمی ویلز» و «لری سنگر» هستند. هم\u200cاکنون بنیاد غیرانتفاعی ویکی\u200cمدیا پروژهٔ ویکی\u200cپدیا را پشتیبانی می\u200cکند. میزبان\u200cهای اینترنتی اصلی این وبگاه در شهر تامپای فلوریدا هستند. همچنین میزبان\u200cهای اضافی دیگری هم در شهرهای آمستردام و سئول به این وبگاه یاری می\u200cرسانند.'}

In [ ]:
import json
from transformers import AutoTokenizer

# Load tokenizer — using jina3 since that's one of your models
tokenizer = AutoTokenizer.from_pretrained("jinaai/jina-embeddings-v3", trust_remote_code=True)

# Extract text field — adjust 'text' to your actual field name
texts = []
for record in data:
    # MIRACL corpus usually has 'title' and 'text' fields
    content = record.get("text", "") or ""
    title = record.get("title", "") or ""
    texts.append(title + " " + content if title else content)

print(f"Sample record keys: {list(data[0].keys())}")
print(f"Sample text: {texts[0][:200]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

configuration_xlm_roberta.py:   0%|          | 0.00/6.54k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- configuration_xlm_roberta.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
`torch_dtype` is deprecated! Use `dtype` instead!


tokenizer_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Sample record keys: ['docid', 'title', 'text']
Sample text: ویکی‌پدیا ویکی‌پدیا (کوته‌نوشت به‌صورت «وپ» و «WP») یک دانشنامه برخط چندزبانه مبتنی بر وب با محتوای آزاد و همکاری باز است که با همکاری افراد داوطلب نوشته می‌شود و هر کسی که به اینترنت و وب دسترسی داشت


In [ ]:
import numpy as np

# Character lengths
char_lengths = [len(t) for t in texts]

# Token lengths (batch for speed)
batch_size = 512
token_lengths = []

for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    encoded = tokenizer(batch, truncation=False, padding=False)
    token_lengths.extend([len(ids) for ids in encoded["input_ids"]])

# Report
print(f"Total documents: {len(texts)}")
print(f"\n--- Character Length ---")
print(f"  Mean:   {np.mean(char_lengths):.1f}")
print(f"  Median: {np.median(char_lengths):.1f}")
print(f"  Max:    {np.max(char_lengths)}")
print(f"  Min:    {np.min(char_lengths)}")

print(f"\n--- Token Length ---")
print(f"  Mean:   {np.mean(token_lengths):.1f}")
print(f"  Median: {np.median(token_lengths):.1f}")
print(f"  Max:    {np.max(token_lengths)}")
print(f"  Min:    {np.min(token_lengths)}")

# Bonus: % of docs exceeding 512 tokens (jina's context limit)
over_512 = sum(1 for l in token_lengths if l > 512)
print(f"\n  Docs > 512 tokens: {over_512} ({100*over_512/len(token_lengths):.1f}%)")

Total documents: 500000

--- Character Length ---
  Mean:   331.1
  Median: 234.0
  Max:    13669
  Min:    5

--- Token Length ---
  Mean:   103.3
  Median: 73.0
  Max:    4258
  Min:    4

  Docs > 512 tokens: 4272 (0.9%)


# Filter Long Documents (more similar to books)

In [ ]:
import json
from tqdm import tqdm

corpus_file = "/content/drive/MyDrive/docs-0.jsonl"

min_length = 600
long_docs = []

print(f"Filtering documents longer than {min_length} characters...")

with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            doc = json.loads(line)
            text = doc.get("text", "")

            if len(text) >= min_length:
                long_docs.append({
                    "id": doc.get("docid"),
                    "title": doc.get("title", ""),
                    "text": text,
                    "length": len(text)
                })

print(f"\nFound {len(long_docs)} documents longer than {min_length} characters")
print(f"Percentage: {len(long_docs)/500000*100:.2f}% of total corpus")

# Show some statistics
if long_docs:
    lengths = [doc["length"] for doc in long_docs]
    print(f"Average length of long docs: {sum(lengths)/len(lengths):.1f} characters")
    print(f"Max length: {max(lengths)} characters")

    # Show sample
    print("\nSample long document:")
    print(long_docs[0]["text"][:500] + "...")

Filtering documents longer than 600 characters...


500000it [00:16, 29554.59it/s]


Found 68611 documents longer than 600 characters
Percentage: 13.72% of total corpus
Average length of long docs: 960.6 characters
Max length: 13646 characters

Sample long document:
در میان تمام زبان‌های ویکی‌پدیا تا دسامبر ۲۰۱۵ (آذر ۱۳۹۴) بیش از ۱۴۱ میلیون صفحه وجود دارد که بیش از سی و هفت میلیون عدد از آن‌ها مقاله است. از زبان‌های مشهور به ترتیب زبان انگلیسی بیش از ۵ میلیون، سوئدی ۲٫۱ میلیون، آلمانی ۱٫۸ میلیون، هلندی ۱٫۸ میلیون، فرانسوی ۱٫۷ میلیون، روسی ۱٫۲ میلیون، ایتالیایی ۱٫۲ میلیون، اسپانیایی ۱٫۲ میلیون، لهستانی ۱٫۱ میلیون، ویتنامی ۱٫۱ میلیون و ژاپنی و پرتغالی و چینی هر کدام بیش از ۸۵۰ هزار مقاله دارند. تعداد کاربران ثبت‌نام شده حدود ۵۹ میلیون نفر است که از میان آنها،...


## save final corpus

In [ ]:
output_file = "/content/drive/MyDrive/long_docs.jsonl"

print(f"\nSaving {len(long_docs)} documents to {output_file}...")

with open(output_file, 'w', encoding='utf-8') as f:
    for doc in tqdm(long_docs):
        f.write(json.dumps(doc, ensure_ascii=False) + '\n')

print(f"Done! Saved to {output_file}")


Saving 68611 documents to /content/drive/MyDrive/long_docs.jsonl...


100%|██████████| 68611/68611 [00:05<00:00, 13427.22it/s]

Done! Saved to /content/drive/MyDrive/long_docs.jsonl


# Load topics and qrels

In [ ]:
import json
from tqdm import tqdm
import pandas as pd

# ===================== CONFIG =====================
corpus_file = "/content/drive/MyDrive/long_docs.jsonl"
qrels_file = "/content/drive/MyDrive/qrels.miracl-v1.0-fa-dev.tsv"
topics_file = "/content/drive/MyDrive/topics.miracl-v1.0-fa-dev.tsv"

output_topics = "/content/drive/MyDrive/filtered_topics.csv"
output_qrels_tsv = "/content/drive/MyDrive/filtered_qrels.tsv"
output_qrels_csv = "/content/drive/MyDrive/filtered_qrels.csv"
# ===================== COUNT TOPICS =====================
with open(topics_file, 'r', encoding='utf-8') as f:
    total_topics = sum(1 for line in f if line.strip())

# ===================== COUNT QRELS =====================
total_qrels = 0
unique_queries_in_qrels = set()
unique_docs_in_qrels = set()

with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                total_qrels += 1
                unique_queries_in_qrels.add(parts[0])
                unique_docs_in_qrels.add(parts[2])

# ===================== REPORT =====================
print(f"""
============== ORIGINAL FILES ==============
  Topics (queries):          {total_topics}
  Qrel pairs:                {total_qrels}
  Unique queries in qrels:   {len(unique_queries_in_qrels)}
  Unique docs in qrels:      {len(unique_docs_in_qrels)}
  Avg docs per query:        {total_qrels / len(unique_queries_in_qrels):.1f}
============================================
""")


============== ORIGINAL FILES ==============
  Topics (queries):          632
  Qrel pairs:                6571
  Unique queries in qrels:   632
  Unique docs in qrels:      6405
  Avg docs per query:        10.4



In [ ]:
import json
from tqdm import tqdm
import pandas as pd

# ===================== CONFIG =====================
corpus_file = "/content/drive/MyDrive/long_docs.jsonl"
qrels_file = "/content/drive/MyDrive/qrels.miracl-v1.0-fa-dev.tsv"
topics_file = "/content/drive/MyDrive/topics.miracl-v1.0-fa-dev.tsv"

output_topics = "/content/drive/MyDrive/filtered_topics.csv"
output_qrels_tsv = "/content/drive/MyDrive/filtered_qrels.tsv"
output_qrels_csv = "/content/drive/MyDrive/filtered_qrels.csv"

# ===================== LOAD FILTERED CORPUS DOC IDS =====================
print("Loading filtered document IDs...")
filtered_doc_ids = set()

with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            doc = json.loads(line)
            doc_id = doc.get("docid") or doc.get("id")
            if doc_id:
                filtered_doc_ids.add(str(doc_id))

print(f"Loaded {len(filtered_doc_ids)} document IDs from filtered corpus\n")

# ===================== LOAD & FILTER QRELS =====================
print("Loading and filtering qrels...")
qrels = {}
valid_queries = set()

with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                docid = parts[2]
                rel = int(parts[3])

                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print(f"Found {len(valid_queries)} queries with at least one relevant document in filtered corpus\n")

# ===================== LOAD TOPICS (QUERIES) =====================
print("Loading topics...")
topics = {}

with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid = parts[0]
                query = parts[1]
                if qid in valid_queries:
                    topics[qid] = query

print(f"Final valid queries for evaluation: {len(topics)}\n")

# ===================== SAVE FILTERED TOPICS =====================
pd.DataFrame([
    {"query_id": qid, "query": topics[qid]} for qid in topics
]).to_csv(output_topics, index=False)
print(f"Filtered topics saved to '{output_topics}'")

# ===================== SAVE FILTERED QRELS (CSV) =====================
filtered_qrels_rows = []
for qid, docids in qrels.items():
    for docid in docids:
        filtered_qrels_rows.append({"query_id": qid, "doc_id": docid})

pd.DataFrame(filtered_qrels_rows).to_csv(output_qrels_csv, index=False)
print(f"Filtered qrels saved to '{output_qrels_csv}'")

# ===================== SAVE FILTERED QRELS (TSV - TREC FORMAT) =====================
with open(output_qrels_tsv, 'w', encoding='utf-8') as f:
    for qid, docids in qrels.items():
        for docid in docids:
            f.write(f"{qid}\t0\t{docid}\t1\n")
print(f"Filtered qrels saved to '{output_qrels_tsv}' (TREC format)")

# ===================== SUMMARY =====================
print(f"""
==================== SUMMARY ====================
  Corpus documents (long):  {len(filtered_doc_ids)}
  Valid queries:             {len(topics)}
  Total qrel pairs:          {len(filtered_qrels_rows)}
=================================================
""")

Loading filtered document IDs...


68611it [00:03, 22535.17it/s]


Loaded 68611 document IDs from filtered corpus

Loading and filtering qrels...


6571it [00:00, 16960.29it/s]


Found 140 queries with at least one relevant document in filtered corpus

Loading topics...
Final valid queries for evaluation: 140

Filtered topics saved to '/content/drive/MyDrive/filtered_topics.csv'
Filtered qrels saved to '/content/drive/MyDrive/filtered_qrels.csv'
Filtered qrels saved to '/content/drive/MyDrive/filtered_qrels.tsv' (TREC format)

==================== SUMMARY ====================
  Corpus documents (long):  68611
  Valid queries:             140
  Total qrel pairs:          174



# Test Embedding

## load new corpus and queries

In [ ]:
print("Loading filtered long documents...")
docs = []
doc_ids = []

with open('/content/drive/MyDrive/docs-0.jsonl', 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            doc = json.loads(line)
            text = doc.get("text", "")
            doc_id = str(doc.get("docid") or doc.get("id"))
            if text and len(text) >= 600:
                docs.append(text)
                doc_ids.append(doc_id)

print(f"Loaded {len(docs)} long documents\n")

# Load topics and qrels
topics_df = pd.read_csv('/content/drive/MyDrive/filtered_topics.csv')
qrels_df = pd.read_csv('/content/drive/MyDrive/filtered_qrels.csv')

print(f"Loaded {len(topics_df)} queries and {len(qrels_df)} qrel pairs\n")

Loading filtered long documents...


500000it [00:05, 87242.64it/s] 

Loaded 68611 long documents

Loaded 140 queries and 174 qrel pairs



## chunking

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


config = {"name": "Medium_450",  "size": 450, "overlap": 70}

# 1. Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['size'],
    chunk_overlap=config['overlap'],
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks = []
chunk_to_doc = []   # To map chunk back to original doc

for i, text in enumerate(tqdm(docs, desc="Chunking")):
    chunks = splitter.split_text(text)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_ids[i]] * len(chunks))

print(f"   Created {len(all_chunks)} chunks")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

Chunking: 100%|██████████| 68611/68611 [00:02<00:00, 31676.89it/s]

   Created 204442 chunks


# jinaa-5-nano

## Generate Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer
import time
import torch

# 2. Generate Embeddings
model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"

model = SentenceTransformer(
    model_name,
    trust_remote_code=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

if torch.cuda.is_available():
    model = model.half()

print("   Generating embeddings...")
start_time = time.time()

embeddings = model.encode(all_chunks, batch_size=16, show_progress_bar=True, normalize_embeddings=True)

elapsed = time.time() - start_time

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.22k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

   Generating embeddings...


Batches:   0%|          | 0/12778 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


In [ ]:
# ===================== TIME STATS =====================
print(f"""
============== TIME ==============
  Total time:        {elapsed:.1f} seconds
  Total time:        {elapsed/60:.2f} minutes
  Chunks per second: {len(all_chunks)/elapsed:.1f}
  Avg per chunk:     {elapsed/len(all_chunks)*1000:.2f} ms
==================================
""")


============== TIME ==============
  Total time:        615.9 seconds
  Total time:        10.27 minutes
  Chunks per second: 331.9
  Avg per chunk:     3.01 ms



In [ ]:
# ===================== SPACE STATS =====================
# Single embedding
single_embedding_bytes = embeddings[0].nbytes
embedding_dim = embeddings.shape[1]

# Full matrix
total_bytes = embeddings.nbytes
total_mb = total_bytes / (1024 ** 2)
total_gb = total_bytes / (1024 ** 3)

print(f"""
============== SPACE ==============
  Embedding dimension:      {embedding_dim}
  Dtype:                    {embeddings.dtype}
  Single embedding size:    {single_embedding_bytes} bytes ({single_embedding_bytes/1024:.2f} KB)
  Total embeddings:         {len(embeddings)}
  Total matrix size:        {total_mb:.2f} MB ({total_gb:.3f} GB)
===================================
""")


============== SPACE ==============
  Embedding dimension:      768
  Dtype:                    float16
  Single embedding size:    1536 bytes (1.50 KB)
  Total embeddings:         204442
  Total matrix size:        299.48 MB (0.292 GB)



# Evaluation

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 3. Evaluation
print("   Evaluating retrieval...")
hits_at_5 = 0
hits_at_10 = 0
total_queries = 0

start_total = time.time()

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid = row["query_id"]
    query = row["query"]

    # Get relevant docs for this query
    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    query_emb = model.encode([query], normalize_embeddings=True)

    # Get top results
    sims = cosine_similarity(query_emb, embeddings)[0]
    top_indices = sims.argsort()[-10:][::-1]

    retrieved_docs = {chunk_to_doc[idx] for idx in top_indices}

    if relevant & retrieved_docs:
        hits_at_5 += 1 if len(relevant & set(chunk_to_doc[idx] for idx in sims.argsort()[-5:][::-1])) > 0 else 0
        hits_at_10 += 1

hit5 = hits_at_5 / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

duration = time.time() - start_total

results_summary = {
    "config": config['name'],
    "chunk_size": config['size'],
    "overlap": config['overlap'],
    "num_chunks": len(all_chunks),
    "avg_chunks_per_doc": round(len(all_chunks) / len(docs), 2),
    "hit@5": round(hit5, 4),
    "hit@10": round(hit10, 4),
    "time_seconds": round(duration, 2)
}

print(f"   Hit@5: {hit5:.4f} | Hit@10: {hit10:.4f} | Time: {duration:.1f}s")

   Evaluating retrieval...


Evaluating:   0%|          | 0/140 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Evaluating:   1%|          | 1/140 [00:04<10:00,  4.32s/it]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Evaluating:   1%|▏         | 2/140 [00:09<11:26,  4.97s/it]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The at

NameError: name 'results_summary' is not defined

In [ ]:
results_summary = {
    "config": config['name'],
    "chunk_size": config['size'],
    "overlap": config['overlap'],
    "num_chunks": len(all_chunks),
    "avg_chunks_per_doc": round(len(all_chunks) / len(docs), 2),
    "hit@5": round(hit5, 4),
    "hit@10": round(hit10, 4),
    "time_seconds": round(duration, 2)
}

print(f"   Hit@5: {hit5:.4f} | Hit@10: {hit10:.4f} | Time: {duration:.1f}s")
print(results_summary)

   Hit@5: 0.6214 | Hit@10: 0.7071 | Time: 338.2s
{'config': 'Medium_450', 'chunk_size': 450, 'overlap': 70, 'num_chunks': 204442, 'avg_chunks_per_doc': 2.98, 'hit@5': 0.6214, 'hit@10': 0.7071, 'time_seconds': 338.15}


# jinaa3

In [ ]:
!pip install -q \
  sentence-transformers==3.3.1 \
  transformers==4.47.1 \
  tokenizers==0.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 103.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.3 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer
import time
import torch

# 2. Generate Embeddings
model_name = "jinaai/jina-embeddings-v3"

model = SentenceTransformer(
    model_name,
    trust_remote_code=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

if torch.cuda.is_available():
    model = model.half()

print("   Generating embeddings...")
start_time = time.time()

embeddings = model.encode(all_chunks, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

elapsed = time.time() - start_time

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- stochastic_depth.py
- mlp.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/xlm-roberta-flash-implementation:
- embedding.py
- rotary.py
- xlm_padding.py
- mha.py
- bloc

config.json:   0%|          | 0.00/192 [00:00<?, ?B/s]

   Generating embeddings...


Batches:   0%|          | 0/6389 [00:00<?, ?it/s]

In [ ]:
# ===================== TIME STATS =====================
print(f"""
============== TIME ==============
  Total time:        {elapsed:.1f} seconds
  Total time:        {elapsed/60:.2f} minutes
  Chunks per second: {len(all_chunks)/elapsed:.1f}
  Avg per chunk:     {elapsed/len(all_chunks)*1000:.2f} ms
==================================
""")


============== TIME ==============
  Total time:        1429.8 seconds
  Total time:        23.83 minutes
  Chunks per second: 143.0
  Avg per chunk:     6.99 ms



In [ ]:
# ===================== SPACE STATS =====================
# Single embedding
single_embedding_bytes = embeddings[0].nbytes
embedding_dim = embeddings.shape[1]

# Full matrix
total_bytes = embeddings.nbytes
total_mb = total_bytes / (1024 ** 2)
total_gb = total_bytes / (1024 ** 3)

print(f"""
============== SPACE ==============
  Embedding dimension:      {embedding_dim}
  Dtype:                    {embeddings.dtype}
  Single embedding size:    {single_embedding_bytes} bytes ({single_embedding_bytes/1024:.2f} KB)
  Total embeddings:         {len(embeddings)}
  Total matrix size:        {total_mb:.2f} MB ({total_gb:.3f} GB)
===================================
""")


============== SPACE ==============
  Embedding dimension:      1024
  Dtype:                    float16
  Single embedding size:    2048 bytes (2.00 KB)
  Total embeddings:         204442
  Total matrix size:        399.30 MB (0.390 GB)



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# 3. Evaluation
print("   Evaluating retrieval...")
hits_at_5 = 0
hits_at_10 = 0
total_queries = 0

start_total = time.time()

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid = row["query_id"]
    query = row["query"]

    # Get relevant docs for this query
    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    query_emb = model.encode([query], normalize_embeddings=True)

    # Get top results
    sims = cosine_similarity(query_emb, embeddings)[0]
    top_indices = sims.argsort()[-10:][::-1]

    retrieved_docs = {chunk_to_doc[idx] for idx in top_indices}

    if relevant & retrieved_docs:
        hits_at_5 += 1 if len(relevant & set(chunk_to_doc[idx] for idx in sims.argsort()[-5:][::-1])) > 0 else 0
        hits_at_10 += 1

hit5 = hits_at_5 / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

duration = time.time() - start_total

results_summary = {
    "config": config['name'],
    "chunk_size": config['size'],
    "overlap": config['overlap'],
    "num_chunks": len(all_chunks),
    "avg_chunks_per_doc": round(len(all_chunks) / len(docs), 2),
    "hit@5": round(hit5, 4),
    "hit@10": round(hit10, 4),
    "time_seconds": round(duration, 2)
}

print(f"   Hit@5: {hit5:.4f} | Hit@10: {hit10:.4f} | Time: {duration:.1f}s")

   Evaluating retrieval...


Evaluating: 100%|██████████| 140/140 [06:49<00:00,  2.92s/it]

   Hit@5: 0.7071 | Hit@10: 0.8571 | Time: 409.4s


# test different chunk sizes

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# config = {"name": "small_300",  "size": 300, "overlap": 50}
# config = {"name": "Medium_450",  "size": 450, "overlap": 70}
# config = {"name": "large_600",  "size": 600, "overlap": 90}
config = {"name": "XLarge_800",  "size": 800, "overlap": 120}

# 1. Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['size'],
    chunk_overlap=config['overlap'],
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks = []
chunk_to_doc = []   # To map chunk back to original doc

for i, text in enumerate(tqdm(docs, desc="Chunking")):
    chunks = splitter.split_text(text)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_ids[i]] * len(chunks))

print(f"   Created {len(all_chunks)} chunks")

# 2. Generate Embeddings
model_name = "jinaai/jina-embeddings-v3"

model = SentenceTransformer(
    model_name,
    trust_remote_code=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

if torch.cuda.is_available():
    model = model.half()

print("   Generating embeddings...")
start_time = time.time()

embeddings = model.encode(all_chunks, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

elapsed = time.time() - start_time

# ===================== TIME STATS =====================
print(f"""
============== TIME ==============
  Total time:        {elapsed:.1f} seconds
  Total time:        {elapsed/60:.2f} minutes
  Chunks per second: {len(all_chunks)/elapsed:.1f}
  Avg per chunk:     {elapsed/len(all_chunks)*1000:.2f} ms
==================================
""")

# ===================== SPACE STATS =====================
# Single embedding
single_embedding_bytes = embeddings[0].nbytes
embedding_dim = embeddings.shape[1]

# Full matrix
total_bytes = embeddings.nbytes
total_mb = total_bytes / (1024 ** 2)
total_gb = total_bytes / (1024 ** 3)

print(f"""
============== SPACE ==============
  Embedding dimension:      {embedding_dim}
  Dtype:                    {embeddings.dtype}
  Single embedding size:    {single_embedding_bytes} bytes ({single_embedding_bytes/1024:.2f} KB)
  Total embeddings:         {len(embeddings)}
  Total matrix size:        {total_mb:.2f} MB ({total_gb:.3f} GB)
===================================
""")

# 3. Evaluation
print("   Evaluating retrieval...")
hits_at_5 = 0
hits_at_10 = 0
total_queries = 0

start_total = time.time()

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid = row["query_id"]
    query = row["query"]

    # Get relevant docs for this query
    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    query_emb = model.encode([query], normalize_embeddings=True)

    # Get top results
    sims = cosine_similarity(query_emb, embeddings)[0]
    top_indices = sims.argsort()[-10:][::-1]

    retrieved_docs = {chunk_to_doc[idx] for idx in top_indices}

    if relevant & retrieved_docs:
        hits_at_5 += 1 if len(relevant & set(chunk_to_doc[idx] for idx in sims.argsort()[-5:][::-1])) > 0 else 0
        hits_at_10 += 1

hit5 = hits_at_5 / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0

duration = time.time() - start_total

results_summary = {
    "config": config['name'],
    "chunk_size": config['size'],
    "overlap": config['overlap'],
    "num_chunks": len(all_chunks),
    "avg_chunks_per_doc": round(len(all_chunks) / len(docs), 2),
    "hit@5": round(hit5, 4),
    "hit@10": round(hit10, 4),
    "time_seconds": round(duration, 2)
}

print(f"   Hit@5: {hit5:.4f} | Hit@10: {hit10:.4f} | Time: {duration:.1f}s")

Chunking: 100%|██████████| 68611/68611 [00:01<00:00, 42165.87it/s]


   Created 115620 chunks


   Generating embeddings...


Batches:   0%|          | 0/3614 [00:00<?, ?it/s]


============== TIME ==============
  Total time:        1480.7 seconds
  Total time:        24.68 minutes
  Chunks per second: 78.1
  Avg per chunk:     12.81 ms


============== SPACE ==============
  Embedding dimension:      1024
  Dtype:                    float16
  Single embedding size:    2048 bytes (2.00 KB)
  Total embeddings:         115620
  Total matrix size:        225.82 MB (0.221 GB)

   Evaluating retrieval...


Evaluating: 100%|██████████| 140/140 [03:44<00:00,  1.60s/it]

   Hit@5: 0.7714 | Hit@10: 0.8643 | Time: 224.7s


# Arabic

In [ ]:
import json
import time
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity

# ===================== CONFIG =====================
corpus_file = "/content/drive/MyDrive/docs-0-arabic.jsonl"
qrels_file  = "/content/drive/MyDrive/qrels.miracl-v1.0-ar-dev.tsv"
topics_file = "/content/drive/MyDrive/topics.miracl-v1.0-ar-dev.tsv"

config = {"name": "large_600", "size": 600, "overlap": 90}
model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"
min_length = 600

# ===================== LOAD & FILTER CORPUS =====================
print(f"Filtering documents longer than {min_length} characters...")
long_docs = []
filtered_doc_ids = set()

with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            doc = json.loads(line)
            text = doc.get("text", "")
            if len(text) >= min_length:
                long_docs.append({
                    "id": doc.get("docid"),
                    "title": doc.get("title", ""),
                    "text": text,
                    "length": len(text)
                })
                filtered_doc_ids.add(str(doc.get("docid")))

print(f"Found {len(long_docs)} documents longer than {min_length} characters")

# ===================== LOAD & FILTER QRELS =====================
print("\nLoading and filtering qrels...")
qrels = {}
valid_queries = set()

with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid   = parts[0]
                docid = parts[2]
                rel   = int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print(f"Found {len(valid_queries)} queries with at least one relevant document in filtered corpus")

# ===================== LOAD TOPICS =====================
print("\nLoading topics...")
topics = {}

with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

print(f"Final valid queries for evaluation: {len(topics)}")

# ===================== BUILD DATAFRAMES =====================
topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])

qrels_rows = [
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
]
qrels_df = pd.DataFrame(qrels_rows)

# ===================== SUMMARY =====================
print(f"""
==================== SUMMARY ====================
  Corpus documents (long):  {len(filtered_doc_ids)}
  Valid queries:             {len(topics)}
  Total qrel pairs:          {len(qrels_rows)}
=================================================
""")

# ===================== PREPARE DOCS =====================
docs    = [doc["text"] for doc in long_docs]
doc_ids = [str(doc["id"]) for doc in long_docs]

print(f"Prepared {len(docs)} documents for chunking")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['size'],
    chunk_overlap=config['overlap'],
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks    = []
chunk_to_doc  = []

for i, text in enumerate(tqdm(docs, desc="Chunking")):
    chunks = splitter.split_text(text)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_ids[i]] * len(chunks))

print(f"Created {len(all_chunks)} chunks")
print(f"Avg chunks per doc: {len(all_chunks)/len(docs):.1f}")

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nLoading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()

# ===================== GENERATE EMBEDDINGS =====================
print(f"\nGenerating embeddings for {len(all_chunks)} chunks...")
start_time = time.time()

embeddings = model.encode(
    all_chunks,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time

print(f"""
============== TIME ==============
  Total time:        {elapsed:.1f} seconds ({elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks)/elapsed:.1f}
  Avg per chunk:     {elapsed/len(all_chunks)*1000:.2f} ms
==================================

============== SPACE ==============
  Embedding dimension:   {embeddings.shape[1]}
  Dtype:                 {embeddings.dtype}
  Single embedding:      {embeddings[0].nbytes} bytes ({embeddings[0].nbytes/1024:.2f} KB)
  Total embeddings:      {len(embeddings)}
  Total matrix size:     {embeddings.nbytes/1024**2:.2f} MB ({embeddings.nbytes/1024**3:.3f} GB)
===================================
""")

# ===================== EVALUATION =====================
print("Evaluating retrieval...")
hits_at_5  = 0
hits_at_10 = 0
total_queries = 0

start_eval = time.time()

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    query_emb = model.encode([query], normalize_embeddings=True)

    # Compute similarities
    sims = cosine_similarity(query_emb, embeddings)[0]
    top10_indices = sims.argsort()[-10:][::-1]
    top5_indices  = sims.argsort()[-5:][::-1]

    retrieved_top10 = {str(chunk_to_doc[idx]) for idx in top10_indices}
    retrieved_top5  = {str(chunk_to_doc[idx]) for idx in top5_indices}

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0
eval_duration = time.time() - start_eval

# ===================== RESULTS =====================
results_summary = {
    "config":            config['name'],
    "chunk_size":        config['size'],
    "overlap":           config['overlap'],
    "num_chunks":        len(all_chunks),
    "avg_chunks_per_doc": round(len(all_chunks) / len(docs), 2),
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "time_seconds":      round(eval_duration, 2)
}

print(f"""
==================== RESULTS ====================
  Config:       {results_summary['config']}
  Chunk size:   {results_summary['chunk_size']} | Overlap: {results_summary['overlap']}
  Num chunks:   {results_summary['num_chunks']}
  Hit@5:        {results_summary['hit@5']}
  Hit@10:       {results_summary['hit@10']}
  Eval time:    {results_summary['time_seconds']}s
=================================================
""")

Filtering documents longer than 600 characters...


500000it [00:09, 54245.04it/s]


Found 76968 documents longer than 600 characters

Loading and filtering qrels...


29197it [00:00, 449733.54it/s]


Found 942 queries with at least one relevant document in filtered corpus

Loading topics...
Final valid queries for evaluation: 942

==================== SUMMARY ====================
  Corpus documents (long):  76968
  Valid queries:             942
  Total qrel pairs:          1269

Prepared 76968 documents for chunking


...

Batches:   0%|          | 0/13077 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



============== TIME ==============
  Total time:        834.7 seconds (13.91 min)
  Chunks per second: 250.7
  Avg per chunk:     3.99 ms

============== SPACE ==============
  Embedding dimension:   768
  Dtype:                 float16
  Single embedding:      1536 bytes (1.50 KB)
  Total embeddings:      209227
  Total matrix size:     306.48 MB (0.299 GB)

Evaluating retrieval...


Evaluating:   0%|          | 0/942 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Evaluating:   0%|          | 1/942 [00:03<50:00,  3.19s/it]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Evaluating:   0%|          | 2/942 [00:05<41:13,  2.63s/it]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The at


==================== RESULTS ====================
  Config:       large_600
  Chunk size:   600 | Overlap: 90
  Num chunks:   209227
  Hit@5:        0.7473
  Hit@10:       0.8068
  Eval time:    2010.63s



# English

In [ ]:
# ===================== CONFIG =====================
corpus_file = "/content/drive/MyDrive/docs-0-english.jsonl"
qrels_file  = "/content/drive/MyDrive/qrels.miracl-v1.0-en-dev.tsv"
topics_file = "/content/drive/MyDrive/topics.miracl-v1.0-en-dev.tsv"

config = {"name": "large_600", "size": 600, "overlap": 90}
model_name = "jinaai/jina-embeddings-v5-text-nano-retrieval"
min_length = 600

# ===================== LOAD & FILTER CORPUS =====================
print(f"Filtering documents longer than {min_length} characters...")
long_docs = []
filtered_doc_ids = set()

with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            doc = json.loads(line)
            text = doc.get("text", "")
            if len(text) >= min_length:
                long_docs.append({
                    "id": doc.get("docid"),
                    "title": doc.get("title", ""),
                    "text": text,
                    "length": len(text)
                })
                filtered_doc_ids.add(str(doc.get("docid")))

print(f"Found {len(long_docs)} documents longer than {min_length} characters")

# ===================== LOAD & FILTER QRELS =====================
print("\nLoading and filtering qrels...")
qrels = {}
valid_queries = set()

with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid   = parts[0]
                docid = parts[2]
                rel   = int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print(f"Found {len(valid_queries)} queries with at least one relevant document in filtered corpus")

# ===================== LOAD TOPICS =====================
print("\nLoading topics...")
topics = {}

with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

print(f"Final valid queries for evaluation: {len(topics)}")

# ===================== BUILD DATAFRAMES =====================
topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])

qrels_rows = [
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
]
qrels_df = pd.DataFrame(qrels_rows)

# ===================== SUMMARY =====================
print(f"""
==================== SUMMARY ====================
  Corpus documents (long):  {len(filtered_doc_ids)}
  Valid queries:             {len(topics)}
  Total qrel pairs:          {len(qrels_rows)}
=================================================
""")

# ===================== PREPARE DOCS =====================
docs    = [doc["text"] for doc in long_docs]
doc_ids = [str(doc["id"]) for doc in long_docs]

print(f"Prepared {len(docs)} documents for chunking")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['size'],
    chunk_overlap=config['overlap'],
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks    = []
chunk_to_doc  = []

for i, text in enumerate(tqdm(docs, desc="Chunking")):
    chunks = splitter.split_text(text)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_ids[i]] * len(chunks))

print(f"Created {len(all_chunks)} chunks")
print(f"Avg chunks per doc: {len(all_chunks)/len(docs):.1f}")

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nLoading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()

# ===================== GENERATE EMBEDDINGS =====================
print(f"\nGenerating embeddings for {len(all_chunks)} chunks...")
start_time = time.time()

embeddings = model.encode(
    all_chunks,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time

print(f"""
============== TIME ==============
  Total time:        {elapsed:.1f} seconds ({elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks)/elapsed:.1f}
  Avg per chunk:     {elapsed/len(all_chunks)*1000:.2f} ms
==================================

============== SPACE ==============
  Embedding dimension:   {embeddings.shape[1]}
  Dtype:                 {embeddings.dtype}
  Single embedding:      {embeddings[0].nbytes} bytes ({embeddings[0].nbytes/1024:.2f} KB)
  Total embeddings:      {len(embeddings)}
  Total matrix size:     {embeddings.nbytes/1024**2:.2f} MB ({embeddings.nbytes/1024**3:.3f} GB)
===================================
""")

# ===================== EVALUATION =====================
print("Evaluating retrieval...")
hits_at_5  = 0
hits_at_10 = 0
total_queries = 0

start_eval = time.time()

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    query_emb = model.encode([query], normalize_embeddings=True)

    # Compute similarities
    sims = cosine_similarity(query_emb, embeddings)[0]
    top10_indices = sims.argsort()[-10:][::-1]
    top5_indices  = sims.argsort()[-5:][::-1]

    retrieved_top10 = {str(chunk_to_doc[idx]) for idx in top10_indices}
    retrieved_top5  = {str(chunk_to_doc[idx]) for idx in top5_indices}

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0
eval_duration = time.time() - start_eval

# ===================== RESULTS =====================
results_summary = {
    "config":            config['name'],
    "chunk_size":        config['size'],
    "overlap":           config['overlap'],
    "num_chunks":        len(all_chunks),
    "avg_chunks_per_doc": round(len(all_chunks) / len(docs), 2),
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "time_seconds":      round(eval_duration, 2)
}

print(f"""
==================== RESULTS ====================
  Config:       {results_summary['config']}
  Chunk size:   {results_summary['chunk_size']} | Overlap: {results_summary['overlap']}
  Num chunks:   {results_summary['num_chunks']}
  Hit@5:        {results_summary['hit@5']}
  Hit@10:       {results_summary['hit@10']}
  Eval time:    {results_summary['time_seconds']}s
=================================================
""")

Filtering documents longer than 600 characters...


500000it [00:14, 33398.98it/s]


Found 162848 documents longer than 600 characters

Loading and filtering qrels...


8350it [00:00, 14784.22it/s]


Found 89 queries with at least one relevant document in filtered corpus

Loading topics...
Final valid queries for evaluation: 89

==================== SUMMARY ====================
  Corpus documents (long):  162848
  Valid queries:             89
  Total qrel pairs:          109

Prepared 162848 documents for chunking


...

Batches:   0%|          | 0/23795 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



============== TIME ==============
  Total time:        990.4 seconds (16.51 min)
  Chunks per second: 384.4
  Avg per chunk:     2.60 ms

============== SPACE ==============
  Embedding dimension:   768
  Dtype:                 float16
  Single embedding:      1536 bytes (1.50 KB)
  Total embeddings:      380707
  Total matrix size:     557.68 MB (0.545 GB)

Evaluating retrieval...


Evaluating:   0%|          | 0/89 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Evaluating:   1%|          | 1/89 [00:05<08:36,  5.87s/it]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Evaluating:   2%|▏         | 2/89 [00:11<08:16,  5.70s/it]/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:202: FutureWarning: The atten


==================== RESULTS ====================
  Config:       large_600
  Chunk size:   600 | Overlap: 90
  Num chunks:   380707
  Hit@5:        0.8315
  Hit@10:       0.8876
  Eval time:    343.18s



# English corpus (jinaa3)

In [4]:
!pip install -q \
  sentence-transformers==3.3.1 \
  transformers==4.47.1 \
  tokenizers==0.21.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 99.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.7 MB/s eta 0:00:00


In [2]:
# ===================== CONFIG =====================
corpus_file = "/content/drive/MyDrive/docs-0-english.jsonl"
qrels_file  = "/content/drive/MyDrive/qrels.miracl-v1.0-en-dev.tsv"
topics_file = "/content/drive/MyDrive/topics.miracl-v1.0-en-dev.tsv"

config = {"name": "large_600", "size": 600, "overlap": 90}
model_name = "jinaai/jina-embeddings-v3"
min_length = 600

# ===================== LOAD & FILTER CORPUS =====================
print(f"Filtering documents longer than {min_length} characters...")
long_docs = []
filtered_doc_ids = set()

with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            doc = json.loads(line)
            text = doc.get("text", "")
            if len(text) >= min_length:
                long_docs.append({
                    "id": doc.get("docid"),
                    "title": doc.get("title", ""),
                    "text": text,
                    "length": len(text)
                })
                filtered_doc_ids.add(str(doc.get("docid")))

print(f"Found {len(long_docs)} documents longer than {min_length} characters")

# ===================== LOAD & FILTER QRELS =====================
print("\nLoading and filtering qrels...")
qrels = {}
valid_queries = set()

with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid   = parts[0]
                docid = parts[2]
                rel   = int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print(f"Found {len(valid_queries)} queries with at least one relevant document in filtered corpus")

# ===================== LOAD TOPICS =====================
print("\nLoading topics...")
topics = {}

with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

print(f"Final valid queries for evaluation: {len(topics)}")

# ===================== BUILD DATAFRAMES =====================
topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])

qrels_rows = [
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
]
qrels_df = pd.DataFrame(qrels_rows)

# ===================== SUMMARY =====================
print(f"""
==================== SUMMARY ====================
  Corpus documents (long):  {len(filtered_doc_ids)}
  Valid queries:             {len(topics)}
  Total qrel pairs:          {len(qrels_rows)}
=================================================
""")

# ===================== PREPARE DOCS =====================
docs    = [doc["text"] for doc in long_docs]
doc_ids = [str(doc["id"]) for doc in long_docs]

print(f"Prepared {len(docs)} documents for chunking")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['size'],
    chunk_overlap=config['overlap'],
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks    = []
chunk_to_doc  = []

for i, text in enumerate(tqdm(docs, desc="Chunking")):
    chunks = splitter.split_text(text)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_ids[i]] * len(chunks))

print(f"Created {len(all_chunks)} chunks")
print(f"Avg chunks per doc: {len(all_chunks)/len(docs):.1f}")

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nLoading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()

# ===================== GENERATE EMBEDDINGS =====================
print(f"\nGenerating embeddings for {len(all_chunks)} chunks...")
start_time = time.time()

embeddings = model.encode(
    all_chunks,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time

print(f"""
============== TIME ==============
  Total time:        {elapsed:.1f} seconds ({elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks)/elapsed:.1f}
  Avg per chunk:     {elapsed/len(all_chunks)*1000:.2f} ms
==================================

============== SPACE ==============
  Embedding dimension:   {embeddings.shape[1]}
  Dtype:                 {embeddings.dtype}
  Single embedding:      {embeddings[0].nbytes} bytes ({embeddings[0].nbytes/1024:.2f} KB)
  Total embeddings:      {len(embeddings)}
  Total matrix size:     {embeddings.nbytes/1024**2:.2f} MB ({embeddings.nbytes/1024**3:.3f} GB)
===================================
""")

# ===================== EVALUATION =====================
print("Evaluating retrieval...")
hits_at_5  = 0
hits_at_10 = 0
total_queries = 0

start_eval = time.time()

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    query_emb = model.encode([query], normalize_embeddings=True)

    # Compute similarities
    sims = cosine_similarity(query_emb, embeddings)[0]
    top10_indices = sims.argsort()[-10:][::-1]
    top5_indices  = sims.argsort()[-5:][::-1]

    retrieved_top10 = {str(chunk_to_doc[idx]) for idx in top10_indices}
    retrieved_top5  = {str(chunk_to_doc[idx]) for idx in top5_indices}

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0
eval_duration = time.time() - start_eval

# ===================== RESULTS =====================
results_summary = {
    "config":            config['name'],
    "chunk_size":        config['size'],
    "overlap":           config['overlap'],
    "num_chunks":        len(all_chunks),
    "avg_chunks_per_doc": round(len(all_chunks) / len(docs), 2),
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "time_seconds":      round(eval_duration, 2)
}

print(f"""
==================== RESULTS ====================
  Config:       {results_summary['config']}
  Chunk size:   {results_summary['chunk_size']} | Overlap: {results_summary['overlap']}
  Num chunks:   {results_summary['num_chunks']}
  Hit@5:        {results_summary['hit@5']}
  Hit@10:       {results_summary['hit@10']}
  Eval time:    {results_summary['time_seconds']}s
=================================================
""")

Filtering documents longer than 600 characters...


500000it [00:07, 64643.01it/s]


Found 162848 documents longer than 600 characters

Loading and filtering qrels...


8350it [00:00, 9417.55it/s]


Found 89 queries with at least one relevant document in filtered corpus

Loading topics...
Final valid queries for evaluation: 89

==================== SUMMARY ====================
  Corpus documents (long):  162848
  Valid queries:             89
  Total qrel pairs:          109

Prepared 162848 documents for chunking


...


Generating embeddings for 380707 chunks...


Batches:   0%|          | 0/23795 [00:00<?, ?it/s]


============== TIME ==============
  Total time:        2403.5 seconds (40.06 min)
  Chunks per second: 158.4
  Avg per chunk:     6.31 ms

============== SPACE ==============
  Embedding dimension:   1024
  Dtype:                 float16
  Single embedding:      2048 bytes (2.00 KB)
  Total embeddings:      380707
  Total matrix size:     743.57 MB (0.726 GB)

Evaluating retrieval...


Evaluating: 100%|██████████| 89/89 [07:42<00:00,  5.20s/it]


==================== RESULTS ====================
  Config:       large_600
  Chunk size:   600 | Overlap: 90
  Num chunks:   380707
  Hit@5:        0.809
  Hit@10:       0.8989
  Eval time:    462.78s



# Arabic corpus (jinaa3)

In [3]:
# ===================== CONFIG =====================
corpus_file = "/content/drive/MyDrive/docs-0-arabic.jsonl"
qrels_file  = "/content/drive/MyDrive/qrels.miracl-v1.0-ar-dev.tsv"
topics_file = "/content/drive/MyDrive/topics.miracl-v1.0-ar-dev.tsv"

config = {"name": "large_600", "size": 600, "overlap": 90}
model_name = "jinaai/jina-embeddings-v3"
min_length = 600

# ===================== LOAD & FILTER CORPUS =====================
print(f"Filtering documents longer than {min_length} characters...")
long_docs = []
filtered_doc_ids = set()

with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            doc = json.loads(line)
            text = doc.get("text", "")
            if len(text) >= min_length:
                long_docs.append({
                    "id": doc.get("docid"),
                    "title": doc.get("title", ""),
                    "text": text,
                    "length": len(text)
                })
                filtered_doc_ids.add(str(doc.get("docid")))

print(f"Found {len(long_docs)} documents longer than {min_length} characters")

# ===================== LOAD & FILTER QRELS =====================
print("\nLoading and filtering qrels...")
qrels = {}
valid_queries = set()

with open(qrels_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        if line.strip():
            parts = line.strip().split()
            if len(parts) >= 4:
                qid   = parts[0]
                docid = parts[2]
                rel   = int(parts[3])
                if rel > 0 and docid in filtered_doc_ids:
                    if qid not in qrels:
                        qrels[qid] = []
                    qrels[qid].append(docid)
                    valid_queries.add(qid)

print(f"Found {len(valid_queries)} queries with at least one relevant document in filtered corpus")

# ===================== LOAD TOPICS =====================
print("\nLoading topics...")
topics = {}

with open(topics_file, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                qid, query = parts[0], parts[1]
                if qid in valid_queries:
                    topics[qid] = query

print(f"Final valid queries for evaluation: {len(topics)}")

# ===================== BUILD DATAFRAMES =====================
topics_df = pd.DataFrame(topics.items(), columns=["query_id", "query"])

qrels_rows = [
    {"query_id": qid, "doc_id": docid}
    for qid, docids in qrels.items()
    for docid in docids
]
qrels_df = pd.DataFrame(qrels_rows)

# ===================== SUMMARY =====================
print(f"""
==================== SUMMARY ====================
  Corpus documents (long):  {len(filtered_doc_ids)}
  Valid queries:             {len(topics)}
  Total qrel pairs:          {len(qrels_rows)}
=================================================
""")

# ===================== PREPARE DOCS =====================
docs    = [doc["text"] for doc in long_docs]
doc_ids = [str(doc["id"]) for doc in long_docs]

print(f"Prepared {len(docs)} documents for chunking")

# ===================== CHUNKING =====================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=config['size'],
    chunk_overlap=config['overlap'],
    separators=["\n\n", "\n", "۔", "؟", "!", ".", " ", ""]
)

all_chunks    = []
chunk_to_doc  = []

for i, text in enumerate(tqdm(docs, desc="Chunking")):
    chunks = splitter.split_text(text)
    all_chunks.extend(chunks)
    chunk_to_doc.extend([doc_ids[i]] * len(chunks))

print(f"Created {len(all_chunks)} chunks")
print(f"Avg chunks per doc: {len(all_chunks)/len(docs):.1f}")

# ===================== LOAD MODEL =====================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nLoading model on {device}...")

model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
if torch.cuda.is_available():
    model = model.half()

# ===================== GENERATE EMBEDDINGS =====================
print(f"\nGenerating embeddings for {len(all_chunks)} chunks...")
start_time = time.time()

embeddings = model.encode(
    all_chunks,
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True
)

elapsed = time.time() - start_time

print(f"""
============== TIME ==============
  Total time:        {elapsed:.1f} seconds ({elapsed/60:.2f} min)
  Chunks per second: {len(all_chunks)/elapsed:.1f}
  Avg per chunk:     {elapsed/len(all_chunks)*1000:.2f} ms
==================================

============== SPACE ==============
  Embedding dimension:   {embeddings.shape[1]}
  Dtype:                 {embeddings.dtype}
  Single embedding:      {embeddings[0].nbytes} bytes ({embeddings[0].nbytes/1024:.2f} KB)
  Total embeddings:      {len(embeddings)}
  Total matrix size:     {embeddings.nbytes/1024**2:.2f} MB ({embeddings.nbytes/1024**3:.3f} GB)
===================================
""")

# ===================== EVALUATION =====================
print("Evaluating retrieval...")
hits_at_5  = 0
hits_at_10 = 0
total_queries = 0

start_eval = time.time()

for _, row in tqdm(topics_df.iterrows(), desc="Evaluating", total=len(topics_df)):
    qid   = row["query_id"]
    query = row["query"]

    relevant = set(qrels_df[qrels_df["query_id"] == qid]["doc_id"].astype(str))
    if not relevant:
        continue

    total_queries += 1

    # Encode query
    query_emb = model.encode([query], normalize_embeddings=True)

    # Compute similarities
    sims = cosine_similarity(query_emb, embeddings)[0]
    top10_indices = sims.argsort()[-10:][::-1]
    top5_indices  = sims.argsort()[-5:][::-1]

    retrieved_top10 = {str(chunk_to_doc[idx]) for idx in top10_indices}
    retrieved_top5  = {str(chunk_to_doc[idx]) for idx in top5_indices}

    if relevant & retrieved_top5:
        hits_at_5 += 1
    if relevant & retrieved_top10:
        hits_at_10 += 1

hit5  = hits_at_5  / total_queries if total_queries > 0 else 0
hit10 = hits_at_10 / total_queries if total_queries > 0 else 0
eval_duration = time.time() - start_eval

# ===================== RESULTS =====================
results_summary = {
    "config":            config['name'],
    "chunk_size":        config['size'],
    "overlap":           config['overlap'],
    "num_chunks":        len(all_chunks),
    "avg_chunks_per_doc": round(len(all_chunks) / len(docs), 2),
    "hit@5":             round(hit5, 4),
    "hit@10":            round(hit10, 4),
    "time_seconds":      round(eval_duration, 2)
}

print(f"""
==================== RESULTS ====================
  Config:       {results_summary['config']}
  Chunk size:   {results_summary['chunk_size']} | Overlap: {results_summary['overlap']}
  Num chunks:   {results_summary['num_chunks']}
  Hit@5:        {results_summary['hit@5']}
  Hit@10:       {results_summary['hit@10']}
  Eval time:    {results_summary['time_seconds']}s
=================================================
""")

Filtering documents longer than 600 characters...


500000it [00:12, 40333.98it/s] 


Found 76968 documents longer than 600 characters

Loading and filtering qrels...


29197it [00:01, 28689.04it/s]


Found 942 queries with at least one relevant document in filtered corpus

Loading topics...
Final valid queries for evaluation: 942

==================== SUMMARY ====================
  Corpus documents (long):  76968
  Valid queries:             942
  Total qrel pairs:          1269

Prepared 76968 documents for chunking


...


Generating embeddings for 209227 chunks...


Batches:   0%|          | 0/13077 [00:00<?, ?it/s]


============== TIME ==============
  Total time:        1739.6 seconds (28.99 min)
  Chunks per second: 120.3
  Avg per chunk:     8.31 ms

============== SPACE ==============
  Embedding dimension:   1024
  Dtype:                 float16
  Single embedding:      2048 bytes (2.00 KB)
  Total embeddings:      209227
  Total matrix size:     408.65 MB (0.399 GB)

Evaluating retrieval...


Evaluating: 100%|██████████| 942/942 [45:32<00:00,  2.90s/it]


==================== RESULTS ====================
  Config:       large_600
  Chunk size:   600 | Overlap: 90
  Num chunks:   209227
  Hit@5:        0.8482
  Hit@10:       0.9013
  Eval time:    2732.62s



In [ ]:
!pip install -q datasets mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 22.0 MB/s eta 0:00:00


In [ ]:
import mteb

tasks = mteb.get_tasks(
    task_types=["Retrieval"],
    languages=["fas"]
)

print(f"Total Persian Retrieval Tasks: {len(tasks)}\n")
print("Top 20 Tasks:\n")

for i, task in enumerate(tasks[:20]):
    print(f"{i+1:2d}. {task.metadata.name}")
    if hasattr(task.metadata, 'main_score'):
        print(f"    Main Score: {task.metadata.main_score}")
    print("-" * 60)

Total Persian Retrieval Tasks: 44

Top 20 Tasks:

 1. CQADupstackRetrieval-Fa
    Main Score: ndcg_at_10
------------------------------------------------------------
 2. ArguAna-Fa.v2
    Main Score: ndcg_at_10
------------------------------------------------------------
 3. CQADupstackAndroidRetrieval-Fa
    Main Score: ndcg_at_10
------------------------------------------------------------
 4. CQADupstackEnglishRetrieval-Fa
    Main Score: ndcg_at_10
------------------------------------------------------------
 5. CQADupstackGamingRetrieval-Fa
    Main Score: ndcg_at_10
------------------------------------------------------------
 6. CQADupstackGisRetrieval-Fa
    Main Score: ndcg_at_10
------------------------------------------------------------
 7. CQADupstackMathematicaRetrieval-Fa
    Main Score: ndcg_at_10
------------------------------------------------------------
 8. CQADupstackPhysicsRetrieval-Fa
    Main Score: ndcg_at_10
----------------------------------------------------

In [ ]:
import mteb
from sentence_transformers import SentenceTransformer
import torch

# ===================== MODELS TO TEST =====================
models_to_test = {
    "Jina-v3": "jinaai/jina-embeddings-v3",
    "Jina-v5-Nano": "jinaai/jina-embeddings-v5-text-nano-retrieval"
}

# ===================== MIRACL TASKS =====================
# Get Persian and Arabic retrieval tasks
tasks = mteb.get_tasks(
    task_types=["Retrieval"],
    languages=["fas", "ara"]   # fas = Persian, ara = Arabic
)

print(f"Found {len(tasks)} MIRACL tasks (Persian + Arabic)\n")

# Optional: Filter to only dev set for faster testing
tasks = [task for task in tasks if "dev" in task.metadata.name.lower()]

print("Selected Tasks:")
for task in tasks:
    print(" -", task.metadata.name)

Found 52 MIRACL tasks (Persian + Arabic)

Selected Tasks:


In [ ]:
!pip install -q mteb sentence-transformers

import mteb
from sentence_transformers import SentenceTransformer
import torch

# ===================== MODELS =====================
models_to_test = {
    "Jina-v3": "jinaai/jina-embeddings-v3",
    "Jina-v5-Nano": "jinaai/jina-embeddings-v5-text-nano-retrieval"
}

# ===================== LOAD MIRACL TASKS ONLY =====================
print("Loading MIRACL tasks...")

tasks = mteb.get_tasks(
    task_types=["Retrieval"],
    languages=["fas", "ara"]   # Persian + Arabic
)

# Filter only MIRACL tasks (important)
tasks = [task for task in tasks if "miracl" in task.metadata.name.lower()]

print(f"✅ Loaded {len(tasks)} MIRACL tasks\n")
for task in tasks:
    print("→", task.metadata.name)

Loading MIRACL tasks...
✅ Loaded 3 MIRACL tasks

→ MIRACLRetrieval
→ MIRACLRetrievalHardNegatives
→ MIRACLRetrievalHardNegatives.v2


In [ ]:
import mteb
from sentence_transformers import SentenceTransformer
import torch

# ===================== MODELS =====================
models_to_test = {
    "Jina-v3": "jinaai/jina-embeddings-v3",
    "Jina-v5-Nano": "jinaai/jina-embeddings-v5-text-nano-retrieval"
}

# ===================== LOAD MIRACL =====================
tasks = mteb.get_tasks(
    task_types=["Retrieval"],
    languages=["fas", "ara"]
)

# Keep only MIRACL tasks
tasks = [t for t in tasks if "miracl" in t.metadata.name.lower()]

print(f"Loaded {len(tasks)} MIRACL tasks for evaluation.\n")

# ===================== EVALUATION =====================
for model_name, model_path in models_to_test.items():
    print(f"\n{'='*75}")
    print(f"🚀 Evaluating: {model_name}")
    print(f"{'='*75}")

    try:
        model = SentenceTransformer(
            model_path,
            trust_remote_code=True,
            device='cuda' if torch.cuda.is_available() else 'cpu'
        )

        if torch.cuda.is_available():
            model = model.half()

        evaluator = mteb.MTEB(tasks=tasks)

        results = evaluator.run(
            model=model,
            output_folder=f"miracl_results/{model_name}",
            batch_size=16,          # Safe for Colab
            overwrite_results=True,
            verbosity=1
        )

        print(f"✅ Successfully finished {model_name}\n")

    except Exception as e:
        print(f"❌ Error with {model_name}: {e}\n")

Loaded 3 MIRACL tasks for evaluation.


🚀 Evaluating: Jina-v3
❌ Error with Jina-v3: '<' not supported between instances of 'str' and 'int'


🚀 Evaluating: Jina-v5-Nano


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/274 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

fa-corpus/dev-00000-of-00003.parquet:   0%|          | 0.00/145M [00:00<?, ?B/s]

fa-corpus/dev-00001-of-00003.parquet:   0%|          | 0.00/137M [00:00<?, ?B/s]

fa-corpus/dev-00002-of-00003.parquet:   0%|          | 0.00/192M [00:00<?, ?B/s]

fa-queries/dev-00000-of-00001.parquet:   0%|          | 0.00/32.5k [00:00<?, ?B/s]

KeyboardInterrupt: 